# Street networks and centrality

**Atlas source reading copy · browser-local**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Urbanity/blob/a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f/book/notebooks/street-networks.ipynb)



Standard-library synthetic network, context, imagery-label and graph exercises; these executable cells do not import Urbanity or download OSM/provider data. Native Urbanity, compiled GIS, imagery and graph-ML examples in Markdown are continuations requiring their own environment and data permissions. Owner-authorized original material has no inferred public repository license.

**Provenance:** `book/notebooks/street-networks.ipynb` at `a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f`; original SHA-256 `d0f5c0a941331043d66bab9c59f86473bdca8a567f87bb60db61b5b116e360a0`. Original owner-authored companion incorporated at repository owner request; Urbanity source repository has no public license grant. Package Urbanity and third-party data retain separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Street networks and centrality

**Question:** which intersections sit on many of the short routes through a neighbourhood? A street network represents intersections as nodes and street segments as edges. This notebook uses a tiny, invented network so its calculations run in JupyterLite; it is not an Urbanity result or a real map.

![Synthetic network sketch](../_static/urbanity-pipeline.svg)

[Launch this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Urbanity/lab/lab/index.html?path=notebooks/street-networks.ipynb) · [Official street-network tutorial](https://urbanity.readthedocs.io/en/latest/tutorials/street-networks/)

## Try it: inspect a street graph

An edge list is a compact way to record which intersections are joined. The example includes a small grid and one diagonal connection.

In [1]:
edges = [("A", "B"), ("B", "C"), ("D", "E"), ("E", "F"),
         ("A", "D"), ("B", "E"), ("C", "F"), ("B", "D")]
nodes = sorted({node for edge in edges for node in edge})
neighbors = {node: set() for node in nodes}
for left, right in edges:
    neighbors[left].add(right)
    neighbors[right].add(left)

print("Intersections:", len(nodes))
print("Street segments:", len(edges))
print("Degree by intersection:", {node: len(neighbors[node]) for node in nodes})

Intersections: 6
Street segments: 8
Degree by intersection: {'A': 2, 'B': 4, 'C': 2, 'D': 3, 'E': 3, 'F': 2}


Degree counts the directly connected streets. To reason about routes, breadth-first search finds the fewest-edge path between two intersections. Real street-network analyses can use distance or travel time as edge weights; fewest edges is only a simple teaching stand-in.

In [2]:
def shortest_path(graph, start, target):
    """Return a fewest-edge path, or None when target is unreachable."""
    queue = [start]
    parent = {start: None}
    for current in queue:
        if current == target:
            break
        for neighbor in sorted(graph[current]):
            if neighbor not in parent:
                parent[neighbor] = current
                queue.append(neighbor)
    if target not in parent:
        return None
    path = []
    current = target
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))

route = shortest_path(neighbors, "A", "F")
print("Fewest-edge route A -> F:", " -> ".join(route))
print("Number of street segments in route:", len(route) - 1)

Fewest-edge route A -> F: A -> B -> C -> F
Number of street segments in route: 3


## How this maps to Urbanity

In a native Urbanity environment, provide a study boundary and request a street network. Its result includes a NetworkX graph and geospatial node/edge tables; options can add centrality and contextual attributes. Consult the current [street-network guide](https://urbanity.readthedocs.io/en/latest/tutorials/street-networks/) before adapting the workflow because data availability and APIs can change.

```python
from urbanity import Map

city = Map(country="Singapore")
city.add_polygon_boundary("path/to/study-area.geojson")
graph, nodes, edges = city.get_street_network(
    bandwidth=100,
    network_type="driving",
    graph_attr=True,
)
```

The browser kernel does not have the GIS stack needed to execute that code. See [native setup](https://urbanity.readthedocs.io/en/latest/installation/).

## More native street-network workflows

Other documented pathways include the dual (edge-based) graph, area-level aggregate statistics, and saving node/edge GeoDataFrames to GeoJSON. These are real geospatial operations and should be run in the native environment.

```python
dual_graph, dual_nodes, dual_edges = city.get_street_network(dual=True)
area_summary = city.get_aggregate_stats("Seattle", get_svi=True)
nodes.to_file("nodes.geojson", driver="GeoJSON")
edges.to_file("edges.geojson", driver="GeoJSON")
```

For exact parameters, supported network types, returned columns, and caveats, refer to the current [official street-network tutorial](https://urbanity.readthedocs.io/en/latest/tutorials/street-networks/).

## Reflect

- Which intersection has the highest degree? Does it also lie on the shortest route above?
- What would change if a route used distance or travel time instead of number of segments?
- How might a disconnected street affect accessibility?